# A / B / C portfolio data

Enter the A ticker list. For each A, discover the same industry and configured related industries, download formation prices, rank candidates by correlation strictly above 0.90, and run OLS, Engle-Granger and residual ADF. Select the first two passing substitutes; otherwise skip A.

**2022-2025: selection, regression and tests. 2026: download and reserve after selection.**

Export one combined Open/Close CSV for all three portfolios plus the SPY benchmark, and one CSV containing A/B, A/C and B/C betas.

## 1. Portfolio A: the selected ten assets

The fixed list below was found by searching current large-cap listings in descending market-cap order. Each A has two substitutes; the three portfolios contain 30 distinct tickers. Formation prices alone determine selection and regression coefficients.

Only `data/portfolios_open_close.csv` and `data/pair_betas.csv` are saved. Search details and diagnostics remain in memory.

In [1]:
PORTFOLIO_A_TICKERS = [
    "NVDA",
    "AAPL",
    "MSFT",
    "TSM",
    "BRK-B",
    "JPM",
    "V",
    "MA",
    "PLTR",
    "ORCL",
]

## 2. Environment and imports

Select the registered `Long Short Tax Saving (.venv)` kernel. To recreate the environment, run these commands from the project directory:

```powershell
uv venv --python 3.10 .venv
uv pip install --python .venv\Scripts\python.exe -r requirements.txt
.venv\Scripts\python.exe -m ipykernel install --user --name long-short-tax-saving --display-name "Long Short Tax Saving (.venv)"
```

Restart the kernel after installing dependencies or updating the local Python modules.

Run the notebook with its project folder as the working directory. All file paths are relative to that folder.

In [2]:
from pathlib import Path
import logging
import pandas as pd
from IPython.display import display

from cointegration import Cointegration
from portfolio import CointegrationConfig, PortfolioConfig, PortfolioBuilder, YahooFinanceSource

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## 3. Configuration

Use 2022-2025 Close prices only. Test candidates in descending order of correlation strictly above 0.90. Engle-Granger and residual ADF both use **fixed lag 1, `autolag=None`**; OLS includes an intercept.

Load the configured industry graph from `industry_relationships.json`. `max_candidates=None` checks the returned candidate pool; prices are downloaded in batches and cached in memory for this run only. `CANDIDATE_OVERRIDES` can supply a historical candidate universe.

Set EG confidence explicitly below: **A/B = 95%, A/C = 95%, B/C = 90%**. The test selects the corresponding sample-specific EG critical value. Ordinary residual ADF keeps its fixed -2.86 threshold.

Include `benchmark_ticker="SPY"` in the same Open/Close CSV over 2022-2026. Download it after selection and regression; it is a benchmark only and is excluded from A/B/C membership and pair tests.

In [3]:
AS_OF_DATE = None  # None: current New York date
CANDIDATE_OVERRIDES = {}

coint_config = CointegrationConfig(
    train_start="2022-01-01",
    train_end="2025-12-31",
    min_observations=750,
    max_lag=1,  # Fixed lag 1; autolag=None for both tests
    check_i1=False,
)

config = PortfolioConfig.from_relationship_file(
    Path("industry_relationships.json"),
    cointegration=coint_config,
    data_end="2026-12-31",
    as_of_date=AS_OF_DATE,
    benchmark_ticker="SPY",  # Market benchmark; excluded from A/B/C selection
    min_correlation=0.90,
    correlation_basis="price",
    ab_confidence_level=0.95,  # EG: use the 5% critical value for A/B
    ac_confidence_level=0.95,  # EG: keep the 5% critical value for A/C
    bc_confidence_level=0.90,  # EG: use the 10% critical value for B/C
    max_candidates=None,
    batch_size=20,
    allow_sector_fallback=False,
    disjoint_portfolios=True,
    output_dir=Path("data"),
)
print("Configured industries:", len(config.related_industries))
print("EG confidence:", {"AB": config.ab_confidence_level,
                         "AC": config.ac_confidence_level,
                         "BC": config.bc_confidence_level})
print("Formation price basis: Close; dividends are excluded from portfolio P&L.")

Configured industries: 71
EG confidence: {'AB': 0.95, 'AC': 0.95, 'BC': 0.9}
Formation price basis: Close; dividends are excluded from portfolio P&L.


## Industry and related-asset discovery

- Read Yahoo's current `industry` label and query peers using `EquityQuery` and `yf.screen`. Normalize punctuation differences between provider labels.
- Expand candidates through the configured graph. Semiconductor equipment, energy, food and utility relationships are supply-chain candidates; financial and insurance relationships are related business exposures.
- These are industry-level relationships, not independently verified company-specific supplier/customer contracts. The JSON preserves sources and scope.
- `result["seed_metadata"]` contains A classifications; `result["candidate_universe"]` records candidate scope and labels in memory.
- Portfolios do not reuse tickers. Known alternative share classes are excluded. These screening rules do not establish wash-sale or straddle treatment.

## Optional: repeat the large-cap search (disabled by default)

Retrieve the current top 300 US exchange listings by market cap. Try A seeds from largest to smallest until ten complete triplets are found or the list is exhausted. Candidate assets still come from the same industry and configured related industries.

Search results remain in memory (`save=False`). Only the final build below writes the two requested CSVs. Download 2026 prices only after selecting the triplets.

In [ ]:
SEARCH_AGAIN = False
search_source = None

if SEARCH_AGAIN:
    search_source = YahooFinanceSource(config)
    large_caps = search_source.get_large_cap_universe(limit=300)
    search_result = PortfolioBuilder(config, source=search_source).build(
        large_caps["ticker"].tolist(), target_count=10, save=False,
    )
    display(search_result["triplets"])
    assert len(search_result["triplets"]) == 10, "Fewer than ten triplets; inspect the result or expand the seed universe."
    PORTFOLIO_A_TICKERS = search_result["ticker_lists"]["A"]
    print("PORTFOLIO_A_TICKERS =", PORTFOLIO_A_TICKERS)

## 4. Build and save two CSVs

Continue to the next A when a complete triplet cannot be found. Rejection reasons and download events remain available in `result` for inspection.

This run updates only `data/portfolios_open_close.csv` and `data/pair_betas.csv`. No intermediate files or disk caches are written.

In [ ]:
builder = PortfolioBuilder(config, source=search_source)
result = builder.build(
    PORTFOLIO_A_TICKERS,
    candidate_overrides=CANDIDATE_OVERRIDES,
)

print(f"Accepted triplets: {len(result['triplets'])}; skipped A tickers: {len(result['skipped'])}")
display(result["triplets"])
display(result["skipped"])


## 5. Portfolio A / B / C objects

Each object has `.tickers`, `.members`, `.data` and `.prices`. Members contain frozen model ratios; daily data include vendor OHLCV; `.prices` contains Close prices.

The existing model convention is A=1, B=beta_AB and C=beta_AC. These are Close-price hedge units, not normalized capital weights or executed share quantities. Whole-share sizing uses these Close slopes with Open fills. Intercepts are stored separately. `anchor_A` links the corresponding assets.

In [ ]:
portfolio_A = result["portfolios"]["A"]
portfolio_B = result["portfolios"]["B"]
portfolio_C = result["portfolios"]["C"]

for portfolio in [portfolio_A, portfolio_B, portfolio_C]:
    print(portfolio.name, portfolio.tickers)
    display(portfolio.members)
    display(portfolio.data.head())


### Selected asset names and classifications

The candidate universe is not restricted to US common stocks. The notes below are for identification only.

GAM, SOR, MCI, BLW and AWF are tagged `closed_end_fund`. Other instruments have not been separately classified here. Official sources: [GAM](https://generalamericaninvestors.com/about/stock-issues), [SOR](https://fpa.com/fund/source-capital/), [MCI](https://mci.barings.com/), [BLW](https://www.blackrock.com/us/individual/products/240191/blackrock-limited-duration-income-trust-aggregate-blw), [AWF](https://www.sec.gov/Archives/edgar/data/906013/000119312526256606/d92104dncsr.htm).

## 6. Combined Open/Close data and Close-based pair betas

- `portfolios_open_close.csv`: one row per date, ticker-first two-level columns containing Open/Close, including SPY. Historical Close data are retained so dynamic regressions can include formation observations.
- `pair_betas.csv`: AB, AC and BC rows per triplet, all with `price_basis="Close"`. OLS includes an intercept and uses 2022-2025 Close only. `inherited_beta` is a separate coefficient-ratio diagnostic.
- `beta` is the fixed formation slope. The saved `quote_beta` is its formation-end snapshot and equals beta at that point. During analysis, dynamic `quote_beta` is refitted in memory from historical Close through the preceding signal date. It is no longer an adjustment-factor conversion.
- EG confidence is configured per pair; 5% and 10% flags and marginal labels remain separate. BC is an additional post-selection diagnostic.

```python
prices = pd.read_csv("data/portfolios_open_close.csv", header=[0, 1], index_col=0, parse_dates=True)
prices["NVDA"]  # Open and Close
```

No dividend cash flows are added to either long/short P&L or SPY returns. Price changes on ex-dividend dates remain in Close. The preview below reloads the two saved CSVs without rerunning the industry search.


In [4]:
open_close = pd.read_csv(
    config.output_dir / "portfolios_open_close.csv",
    header=[0, 1], index_col=0, parse_dates=True,
)
pair_betas = pd.read_csv(config.output_dir / "pair_betas.csv")

display(open_close.head())
display(open_close[config.benchmark_ticker].tail())
display(pair_betas[["anchor_A", "pair", "y_ticker", "x_ticker", "alpha", "beta", "quote_beta", "price_basis",
                    "confidence_level", "eg_critical_value", "eg_passed", "eg_pvalue", "cointegration_status"]])

assert len(pair_betas) == 3 * pair_betas["anchor_A"].nunique()
assert pair_betas.groupby("anchor_A")["pair"].apply(set).map(lambda pairs: pairs == {"AB", "AC", "BC"}).all()
assert pair_betas["accepted_for_research"].all(), "Review failed pair diagnostics."
assert pair_betas["used_lag"].eq(1).all()
assert (config.benchmark_ticker, "Open") in open_close.columns
assert (config.benchmark_ticker, "Close") in open_close.columns
assert config.benchmark_ticker not in set(pair_betas["y_ticker"]) | set(pair_betas["x_ticker"])
assert open_close.index.is_unique
assert open_close.columns.names == ["Ticker", "Price"]
print("Trading dates:", len(open_close), "Price columns:", len(open_close.columns), "Pair beta rows:", len(pair_betas))
assert pair_betas["price_basis"].eq("Close").all()


Ticker           NVDA                   AAPL                    MSFT  \
Price            Open      Close        Open       Close        Open   
Date                                                                   
2022-01-03  29.815001  30.121000  177.830002  182.009995  335.350006   
2022-01-04  30.277000  29.290001  182.630005  179.699997  334.829987   
2022-01-05  28.948999  27.604000  179.610001  174.919998  325.859985   
2022-01-06  27.639999  28.177999  172.699997  172.000000  313.149994   
2022-01-07  28.141001  27.247000  172.889999  172.169998  314.149994   

Ticker                         TSM                   BRK-B              ...  \
Price            Close        Open       Close        Open       Close  ...   
Date                                                                    ...   
2022-01-03  334.750000  124.129997  128.800003  300.100006  300.790009  ...   
2022-01-04  329.010010  130.869995  133.399994  301.649994  308.529999  ...   
2022-01-05  316.380005  130.710007  127.059998  309.869995  309.920013  ...   
2022-01-06  313.880005  127.000000  128.470001  312.980011  313.220001  ...   
2022-01-07  314.040009  126.550003  123.500000  315.559998  319.779999  ...   

Ticker           SEIC             BAM               NET              \
Price            Open      Close Open Close        Open       Close   
Date                                                                  
2022-01-03  61.299999  61.150002  NaN   NaN  131.070007  126.160004   
2022-01-04  61.959999  62.919998  NaN   NaN  124.519997  114.959999   
2022-01-05  63.520000  62.360001  NaN   NaN  113.500000  105.949997   
2022-01-06  62.799999  63.270000  NaN   NaN  103.519997  104.040001   
2022-01-07  63.459999  63.330002  NaN   NaN  102.870003  107.000000   

Ticker            GWRE                     SPY              
Price             Open       Close        Open       Close  
Date                                                        
2022-01-03  113.809998  113.290001  476.299988  477.709991  
2022-01-04  113.500000  112.919998  479.220001  477.549988  
2022-01-05  111.769997  107.209999  477.160004  468.380005  
2022-01-06  106.639999  107.239998  467.890015  467.940002  
2022-01-07  107.309998  106.389999  467.950012  466.089996  

[5 rows x 62 columns]

Price,Open,Close
Date,,
2026-10-14,774.919983,769.599976
2026-10-15,766.250000,771.789978
2026-10-16,773.239990,763.250000
2026-10-19,767.229980,766.479980
2026-10-20,765.859985,769.049988


,anchor_A,pair,y_ticker,x_ticker,alpha,beta,quote_beta,price_basis,confidence_level,eg_critical_value,eg_passed,eg_pvalue,cointegration_status
0,NVDA,AB,NVDA,ANET,-26.080030,1.563592,1.563592,Close,0.95,-3.342235,True,0.009752,passed_5pct
1,NVDA,AC,NVDA,NVMI,-48.851525,0.781103,0.781103,Close,0.95,-3.342235,True,0.011376,passed_5pct
2,NVDA,BC,ANET,NVMI,-11.953617,0.483638,0.483638,Close,0.90,-3.048685,True,0.011211,passed_5pct
3,AAPL,AB,AAPL,DAKT,130.460278,6.279021,6.279021,Close,0.95,-3.342235,True,0.001362,passed_5pct
4,AAPL,AC,AAPL,BELFA,128.100684,1.005278,1.005278,Close,0.95,-3.342235,True,0.005610,passed_5pct
5,AAPL,BC,DAKT,BELFA,0.196881,0.151049,0.151049,Close,0.90,-3.048685,True,0.092200,marginal_10pct
6,MSFT,AB,MSFT,UBER,165.153029,3.561458,3.561458,Close,0.95,-3.342235,True,0.008611,passed_5pct
7,MSFT,AC,MSFT,CDNS,53.441146,1.274181,1.274181,Close,0.95,-3.342235,True,0.032055,passed_5pct
8,MSFT,BC,UBER,CDNS,-27.135621,0.340571,0.340571,Close,0.90,-3.048685,True,0.034563,passed_5pct
9,TSM,AB,TSM,AVGO,50.326848,0.667354,0.667354,Close,0.95,-3.342235,True,0.013820,passed_5pct


Trading dates: 1204 Price columns: 62 Pair beta rows: 30


## 7. Formation and reserved-data checks

2026 prices do not affect selection, test statistics, ranking or betas. Save only available 2026 data as of the run date. Missing reserved data are reported without reselecting assets. No 2026 trading performance is calculated here.

In [ ]:
train_prices = result["train_prices"]
prices_2026 = result["holdout_prices"]

assert train_prices.empty or train_prices.index.max() <= pd.Timestamp("2025-12-31")
assert prices_2026.empty or (
    prices_2026.index.min() >= pd.Timestamp("2026-01-01")
    and prices_2026.index.max() <= pd.Timestamp("2026-12-31")
)
for label, frame in [("2022–2025 formation", train_prices), ("2026 reserved", prices_2026)]:
    print(label, frame.shape,
          str(frame.index.min().date()) if len(frame) else "empty",
          str(frame.index.max().date()) if len(frame) else "empty")
display(pd.DataFrame(result["metadata"]["holdout_availability"]))


triplets = result["triplets"]
assert len(triplets) == 10, "Expected ten triplets; inspect skipped seeds and source events."
all_tickers = sum(result["ticker_lists"].values(), [])
assert len(all_tickers) == len(set(all_tickers)) == 30
for prefix in ("AB", "AC"):
    confidence_level = config.ab_confidence_level if prefix == "AB" else config.ac_confidence_level
    assert (triplets[prefix + "_confidence_level"] == confidence_level).all()
    assert (triplets[prefix + "_correlation"] > 0.90).all()
    assert (triplets[prefix + "_test_statistic"] < triplets[prefix + "_critical_value"]).all()
    assert (triplets[prefix + "_residual_adf_test_statistic"] < -2.86).all()
    assert (triplets[prefix + "_used_lag"] == 1).all()
    assert (triplets[prefix + "_residual_adf_used_lag"] == 1).all()
print("Verified: 10 triplets, 30 distinct tickers, correlation > 0.90, EG + ADF passed, fixed lag 1.")

## 8. In-memory diagnostics

Inspect rejection reasons when no match is found. Diagnostic tables are displayed here, without exporting additional files. Do not use 2026 outcomes to repeatedly relax formation-period screening rules.

In [ ]:
display(result["candidate_correlations"].sort_values(
    ["A", "correlation"], ascending=[True, False],
).head(20))
diagnostics = result["diagnostics"]
if not diagnostics.empty:
    display(diagnostics.groupby("reason", dropna=False).size().rename("count").reset_index())

assert set(result["paths"]) == {"portfolios_open_close", "pair_betas"}
display(pd.DataFrame({"file": list(result["paths"]),
                      "path": [str(path) for path in result["paths"].values()]}))

In [ ]:
# Ordinary ADF uses -2.86; EG uses its own cointegration critical values.
critical_columns = ["seed", "y", "x", "correlation", "intercept", "hedge_ratio", "ols_r_squared",
                   "test_statistic", "confidence_level", "critical_value", "cointegrated", "critical_value_5pct", "cointegrated_95", "used_lag",
                   "residual_adf_test_statistic", "residual_adf_critical_value_5pct",
                   "residual_adf_stationary_95", "residual_adf_used_lag"]
display(result["diagnostics"].reindex(columns=critical_columns))

## Data and interpretation limits

- Current Yahoo classifications, market-cap ordering and available securities introduce survivorship and universe look-ahead bias. Historical prices may be revised. Isolating 2026 prices is not a complete point-in-time backtest. Use `CANDIDATE_OVERRIDES` for a historically known universe.
- All formation statistics and betas use `Close`. The combined Open/Close CSV preserves vendor fields from `auto_adjust=False`. These fields do not replace executions, tax lots or cost-basis records.
- Statistical screening does not establish wash-sale or straddle eligibility.
- EG thresholds follow the pair-specific confidence settings (defaults: A/B 95%, A/C 95%, B/C 90%). Ordinary residual ADF remains < -2.86. B/C diagnostics separately label `passed_5pct`, `marginal_10pct`, or `failed`. This research run retains the marginal B/C result without changing its 5% rejection flag.

References: [yfinance screener](https://ranaroussi.github.io/yfinance/reference/api/yfinance.screen.html), [yfinance downloads](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html), [statsmodels cointegration](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html).